## Predictive Modeling with scikit-learn


Training a model is the next step after getting a dataset cleaned and preprocessed. In scikit-learn, most estimators follow the same high-level workflow, which makes it easier to switch between algorithms and compare their behavior.


The scikit-learn estimator API is consistent. In most projects, you move through the following steps:

```mermaid
flowchart LR
    A[Import estimator] --> B[Instantiate model]
    B --> C[Fit on training data]
    C --> D[Predict on unseen data]
    D --> E[Evaluate the results]
```

A minimal example looks like this:

```python
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
```

`fit` learns patterns from the training data, while `predict` applies the trained model to new examples.


## Model Training


To start the modeling process, we need to choose an appropriate algorithm. Whether it is a decision tree, linear regression, or K-nearest neighbors, scikit-learn offers a wide range of models for different problem types. In this notebook, we work on a classification problem.

**Classification** means predicting a category or label, for example whether an email is spam or not spam.

We will use **logistic regression**, a simple and interpretable baseline model for binary classification. Despite its name, logistic regression is used for classification because it models the probability of belonging to a class by applying a sigmoid function to a linear combination of the input features.


Let's start by importing the libraries we need.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split

Now we load the water quality dataset.


In [ ]:
df = pd.read_csv("data/water-quality.csv")
df.head()

The target column `is_safe` indicates whether a water sample is safe to drink. Before training, it is always useful to inspect the class distribution.


In [ ]:
df["is_safe"].value_counts()

The dataset is clearly imbalanced: there are many more unsafe samples than safe ones. That matters because a model can appear strong on accuracy alone while still performing poorly on the minority class. We keep the imbalance here on purpose because later notebooks focus on evaluation metrics in more detail.


Next, we separate the feature columns from the target column.


In [ ]:
X = df.drop(columns="is_safe")
y = df["is_safe"]

Before training, we split the data into a training set and a test set. The model learns only from the training data, while the test data is kept aside for the final evaluation.

Because the target is imbalanced, we use `stratify=y` so that both splits keep a similar class distribution.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=1,
    stratify=y,
)

For models such as logistic regression, scaling can be helpful because the optimization procedure behaves better when features are on similar scales. For simplicity, we skip scaling here, but in a real project you would usually combine preprocessing and modeling in a scikit-learn `Pipeline`.

Now we instantiate the model and train it with `fit`. We increase `max_iter` to give the optimizer enough iterations to converge.


In [ ]:
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)

After fitting, the model has learned one coefficient per feature and an intercept. These values describe how each feature influences the predicted log-odds of the positive class.


In [ ]:
coef_df = pd.DataFrame(
    {
        "feature": X_train.columns,
        "coefficient": logreg.coef_[0],
    }
).sort_values("coefficient", key=lambda s: s.abs(), ascending=False)

coef_df.head(10)

Logistic regression predicts probabilities first and then converts them into class labels. By default, scikit-learn uses a threshold of `0.5` for binary classification.


In [ ]:
predicted_probabilities = pd.DataFrame(
    {
        "prob_safe": logreg.predict_proba(X_test)[:, 1],
    }
)

predicted_probabilities.head()

We can now create class predictions for the unseen test set.


In [ ]:
y_pred = logreg.predict(X_test)

Looking at the predicted class distribution helps us understand the behavior of the classifier. On imbalanced data, many models predict the majority class more often than the minority class.

In [ ]:
pd.Series(y_pred, name="predicted_class").value_counts().rename(
    index={0: "unsafe", 1: "safe"}
)

To get a first impression of model quality, we inspect the accuracy, the confusion matrix, and the classification report. Because the classes are imbalanced, the confusion matrix and the per-class metrics are more informative than accuracy alone.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f}")
print()
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
pd.DataFrame(
    cm,
    index=["actual_unsafe", "actual_safe"],
    columns=["pred_unsafe", "pred_safe"],
)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=logreg.classes_)
disp.plot(cmap="Blues")
plt.show()

A plotted confusion matrix gives the same information as the table, but it is often easier to scan visually because the diagonal highlights correct predictions and the off-diagonal cells show the mistakes.


This notebook showed the basic estimator workflow in scikit-learn: import, instantiate, fit, predict, and evaluate. In the next notebooks, we will look more closely at evaluation metrics and how to improve models through preprocessing and hyperparameter tuning.
